In [3]:
import pandas as pd
import numpy as np
from lets_plot import *
LetsPlot.setup_html()

/tmp/ipykernel_2257360/4262593117.py:1: DeprecationWarning: 
Pyarrow will become a required dependency of pandas in the next major release of pandas (pandas 3.0),
(to allow more performant data types, such as the Arrow string type, and better interoperability with other libraries)
but was not found to be installed on your system.
If this would cause problems for you,
please provide us feedback at https://github.com/pandas-dev/pandas/issues/54466
        
  import pandas as pd


In [4]:
chunks = 100
chromosome = 3
df = pd.DataFrame()

In [5]:
for chunk in range(1, chunks):
    tmp = pd.read_csv("CheckMaskedMapping/Chr" + str(chromosome) + "_Ancestral_Chunk" + str(chunk) + ".INFO")
    df = pd.concat([df, tmp])

In [6]:
df.head()

,Pos,INFO,Major,Minor,Prob,MajorOutgroup,Ancestral,MappedAncestral
0,49997,Major=G;Minor=A;Prob=0.977497;MajorOutgroup=G,G,A,0.977497,G,G,G
1,51185,Major=G;Minor=A;Prob=0.999753;MajorOutgroup=G,G,A,0.999753,G,G,G
2,54981,Major=G;Minor=A;Prob=0.996458;MajorOutgroup=G,G,A,0.996458,G,G,G
3,66485,Major=C;Minor=T;Prob=0.999638;MajorOutgroup=C,C,T,0.999638,C,C,C
4,66871,Major=G;Minor=A;Prob=0.998217;MajorOutgroup=G,G,A,0.998217,G,G,G


In [7]:
info = pd.read_table("../Data/InfoChr" + str(chromosome) + "_SNPs.INFO", sep=" ")
info.head()


,CHROM,POS,REF,ALT,AC,AF
0,3,43849,C,G,1,0.000753
1,3,45090,A,T,6,0.004518
2,3,45140,C,T,6,0.004518
3,3,45149,C,T,6,0.004518
4,3,45156,C,T,44,0.033133


In [8]:
info.shape

(724991, 6)

In [9]:
df = pd.merge(df, info, how = "left", left_on = "Pos", right_on="POS")
df.loc[:, "AF_Ancestral"] = [1 - df.AF[x] if df.REF[x] == df.Ancestral[x] else df.AF[x] for x in range(len(df))]
df.loc[:, "Match"] = df.Ancestral == df.MappedAncestral
df.head()

,Pos,INFO,Major,Minor,Prob,MajorOutgroup,Ancestral,MappedAncestral,CHROM,POS,REF,ALT,AC,AF,AF_Ancestral,Match
0,49997,Major=G;Minor=A;Prob=0.977497;MajorOutgroup=G,G,A,0.977497,G,G,G,3,49997,G,A,6,0.004518,0.995482,True
1,51185,Major=G;Minor=A;Prob=0.999753;MajorOutgroup=G,G,A,0.999753,G,G,G,3,51185,G,A,1,0.000753,0.999247,True
2,54981,Major=G;Minor=A;Prob=0.996458;MajorOutgroup=G,G,A,0.996458,G,G,G,3,54981,G,A,5,0.003765,0.996235,True
3,66485,Major=C;Minor=T;Prob=0.999638;MajorOutgroup=C,C,T,0.999638,C,C,C,3,66485,C,T,1,0.000753,0.999247,True
4,66871,Major=G;Minor=A;Prob=0.998217;MajorOutgroup=G,G,A,0.998217,G,G,G,3,66871,G,A,5,0.003765,0.996235,True


In [10]:
ggplot() + geom_density(aes(x = "AF_Ancestral"), data = df) + \
    facet_wrap(facets = "Match", scales ="free")

In [11]:
df.loc[:, "AncestralType"] = ["MinorAncestral" if df.Minor[x] == df.Ancestral[x] else "MajorAncestral" for x in range(len(df))]
df.AncestralType.value_counts()

AncestralType
MajorAncestral    46912
MinorAncestral     1103
Name: count, dtype: int64

In [12]:
df.Match.value_counts()
print("Percentage of all sites correctly mapped")
print(sum(df.Match) / len(df))
print("#######################################")
print("Percentage of sites where minor is the ancestral")
print(len(df[df.AncestralType == "MinorAncestral"]) / len(df))
print("Percentage of sites where major is the ancestral")
print(len(df[df.AncestralType == "MajorAncestral"]) / len(df))
print("#######################################")
print("Percentage of minor ancestrals correctly mapped")
print(sum(df[df.AncestralType == "MinorAncestral"].Match) / len(df[df.AncestralType == "MinorAncestral"]))
print("Percentage of major ancestrals correctly mapped")
print(sum(df[df.AncestralType == "MajorAncestral"].Match) / len(df[df.AncestralType == "MajorAncestral"]))



Percentage of all sites correctly mapped
0.9343746745808601
#######################################
Percentage of sites where minor is the ancestral
0.022971987920441528
Percentage of sites where major is the ancestral
0.9770280120795585
#######################################
Percentage of minor ancestrals correctly mapped
0.38803263825929285
Percentage of major ancestrals correctly mapped
0.9472203274215553


In [13]:
ggplot() + geom_histogram(aes(x = "AF_Ancestral", fill = "Match"), data = df, posiiton = "dodge") 
#+      facet_wrap(facets = "AncestralType", scales = "free") 

In [14]:
ggplot() + geom_density(aes(x = "AF_Ancestral"), data = df, posiiton = "dodge") +  \
    facet_wrap(facets = "AncestralType", scales = "free") 

In [15]:
# Plot binned ratio distribution of MINOR AS ANCESTRALs correctly mapped
breakpoints = np.linspace(0, 1, 100)

groupsMinorAA = df[df.AncestralType == "MinorAncestral"].groupby(['Match', pd.cut(df.AF, breakpoints)])
minorAACount = groupsMinorAA.size().unstack()

groupsAll = df.groupby(['Match', pd.cut(df.AF, breakpoints)])
allCount = groupsAll.size().unstack()
allCount.head()

minorAACountL = minorAACount.melt()
minorAACountL.loc[:, "Match"] = ["False", "True"] * int(len(minorAACountL) / 2)
minorAACountL.columns = ["AF", "CountMinorAA", "Match"]
allCountL = allCount.melt()
allCountL.loc[:, "Match"] = ["False", "True"] * int(len(allCountL) / 2)
allCountL.columns = ["AF", "CountAll", "Match"]
ratioMatch = allCountL.merge(minorAACountL, on = ["AF", "Match"])
ratioMatch.loc[:, "Ratio"] = ratioMatch.CountMinorAA / ratioMatch.CountAll
ratioMatch.head()

ggplot() + geom_point(aes(x = "AF", y = "Ratio"), data = ratioMatch) + facet_wrap(facets="Match") + \
    ggtitle("Minor as ancestral mapping")

/tmp/ipykernel_2257360/2753796690.py:4: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  groupsMinorAA = df[df.AncestralType == "MinorAncestral"].groupby(['Match', pd.cut(df.AF, breakpoints)])
/tmp/ipykernel_2257360/2753796690.py:7: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  groupsAll = df.groupby(['Match', pd.cut(df.AF, breakpoints)])


In [16]:
# Plot distribution of binned ratio of correctly vs uncorrectly matched mutations for major and minor as ancestral
breakpoints = np.linspace(0, 1, 100)

groupsMatch = df[df.Match == True].groupby(['AncestralType', pd.cut(df.AF_Ancestral, breakpoints)])
matchCount = groupsMatch.size().unstack()

groupsAll = df.groupby(['AncestralType', pd.cut(df.AF_Ancestral, breakpoints)])
allCount = groupsAll.size().unstack()
allCount.head()

matchCountL = matchCount.melt()
matchCountL.loc[:, "AncestralTyoe"] = ["MajorAncestral", "MinorAncestral"] * int(len(matchCountL) / 2)
matchCountL.columns = ["AF_Ancestral", "CountMatch", "AncestralType"]
allCountL = allCount.melt()
allCountL.loc[:, "AncestralTyoe"] = ["MajorAncestral", "MinorAncestral"] * int(len(allCountL) / 2)
allCountL.columns = ["AF_Ancestral", "CountAll", "AncestralType"]
ratioMatch = allCountL.merge(matchCountL, on = ["AF_Ancestral", "AncestralType"])
ratioMatch.loc[:, "Ratio"] = ratioMatch.CountMatch / ratioMatch.CountAll
ratioMatch.head()

ggplot() + geom_point(aes(x = "AF_Ancestral", y = "Ratio"), data = ratioMatch) + \
    ggtitle("Ratio of correctly inferred ancestral state")


/tmp/ipykernel_2257360/4203494462.py:4: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  groupsMatch = df[df.Match == True].groupby(['AncestralType', pd.cut(df.AF_Ancestral, breakpoints)])
/tmp/ipykernel_2257360/4203494462.py:7: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  groupsAll = df.groupby(['AncestralType', pd.cut(df.AF_Ancestral, breakpoints)])
